kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import anndata as ad
import pandas as pd
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "dominguez"
OLIGO_SERIES = SERIES + "_oligo"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

oligo_adata = ad.read_h5ad(os.path.join(clustered_h5ad_dir, OLIGO_SERIES + ".h5ad"))

In [4]:
cell_marker_genes = {
    "Query": ["Mef2a", "Igsf11", "Tcp11l2", "Eml2",  "Lama2", "Svep1", "Ror1", "Ltbp1", "Dysf", "Plxdc2", 
              "Slc5a11", "Acsbg1", "Actn2", "Hivep3", "Dpysl5", "Lrrc63", "Pcsk6",  ],
    "OPC": ["Pdgfra", "Cdo1", "Fabp7", "Hes5", "Bmper", "Rgcc", "Dgkb", "Sgcz", "Chl1", "Ptpn5", "Rlbp1", "Unc13c", "Cspg5", "Calb2", "Pnlip"],
    "COP": ["Sox6", "Gpr17", "Neu4", "Bmp4", "Mycl", "Ust", "Epb41l2", "Enpp6", "Map1b" ],
    "NFOL": ["9630013A20Rik", "Tcf7l2", "Rasgef1b", "Frmd4a", "Pik3r3", "Pdgfa", "Peli1", "Dock6", "Arpc1b", "Ddc", "Rgs16", "Dbx2", "Cmtm7", "Gsto1",  "Rras2", "Cntn1", "Sema3e", "Kank2", "Adamts4", "Etv6", "Gas7", "Sema4d", "Hmgcs1", "Tnik"],
    "MFOL": ["Man1a", "Bmp1", "Mgat5", "Ctps1", "Jcad", "Arhgef19", "Btbd16", "Nherf2", "Cxcl12", "Opalin", ], 
    "MOL": ["Car2", "Apod", "Hapln2", "Anln", "Spock3", "Rasgrp3", "Serpinb1a", "Il33", "Mog", "Mobp", "Mag", "Cnp", "Myrf", "Mbp", ],
}

plot.plot_dotplot(oligo_adata, OLIGO_SERIES, cell_marker_genes, group = "cell")

In [3]:
leiden_to_celltype = {
    "0": "MOL",
    "1": "MOL",
    "2": "OPC",
    "3": "OPC",
    "4": "MFOL",
    "5": "OPC",
    "6": "OPC",
    "7": "MOL",
    "8": "COP",
    "9": "OPC",
    "10": "MFOL",
    "11": "NFOL",
    "12": "OPC",
    "13": "OPC",
    "14": "OPC",
    "15": "MFOL",
    "16": "MOL",
    "17": "MFOL",
    "18": "MOL",
    "19": "OPC",
    "20": "COP",
    "21": "COP",
    "22": "MOL",
    "23": "MOL",
    "24": "COP",
    "25": "MOL",
    "26": "MOL",
    "27": "OPC",
    "28": "NFOL",
}

leiden_str = oligo_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
oligo_adata.obs["cell"] = mapped

In [8]:
oligo_adata.obs["celltype"] = oligo_adata.obs["cell"].copy() #type: ignore
plot.plot_umap(oligo_adata, OLIGO_SERIES, has_celltype=True)